# Library reader survey explorer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SURVEY_COLUMNS_LIST = ["READER_ID", "AGE_BAND", "MEMBERSHIP", "BOOKS_PER_MONTH", "MINUTES_PER_DAY", "FAVOURITE_GENRE", "AUDIOBOOK_SHARE", "BRANCH_VISITS", "SATISFACTION", "EBOOK_DEVICE"]

In [ ]:
def create_sample_reader_df(reader_count=60, seed=21):
    """Build a synthetic reader survey with realistic gaps and codes.

    Every column in SURVEY_COLUMNS_LIST is filled; a few values are
    missing or use the survey's special codes (-1 = refused, 999 = unknown).
    """
    rng = np.random.default_rng(seed)
    age_bands = ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    memberships = ["Basic", "Plus", "Family", "Student"]
    genres = ["Fiction", "Mystery", "Science", "History", "Poetry", "Travel", "Comics"]
    devices = ["None", "Tablet", "E-reader", "Phone"]
    records = []
    for index in range(reader_count):
        age_band = age_bands[rng.integers(0, len(age_bands))]
        membership = memberships[rng.integers(0, len(memberships))]
        base_books = {"16-24": 2.5, "25-34": 2.0, "35-44": 1.8, "45-54": 2.2, "55-64": 3.0, "65+": 3.6}[age_band]
        books = max(0.0, round(rng.normal(base_books, 1.1), 1))
        minutes = int(max(0, rng.normal(25 + books * 9, 12)))
        genre = genres[rng.integers(0, len(genres))]
        audiobook_share = round(float(np.clip(rng.normal(0.25, 0.18), 0, 1)), 2)
        visits = int(rng.poisson(2.5 if membership != "Student" else 4.0))
        satisfaction = int(np.clip(round(rng.normal(3.8, 0.9)), 1, 5))
        device = devices[rng.integers(0, len(devices))]
        if rng.random() < 0.06:
            books = np.nan
        if rng.random() < 0.05:
            minutes = -1
        if rng.random() < 0.04:
            visits = 999
        if rng.random() < 0.05:
            genre = None
        records.append({
            "READER_ID": f"R{1000 + index}",
            "AGE_BAND": age_band,
            "MEMBERSHIP": membership,
            "BOOKS_PER_MONTH": books,
            "MINUTES_PER_DAY": minutes,
            "FAVOURITE_GENRE": genre,
            "AUDIOBOOK_SHARE": audiobook_share,
            "BRANCH_VISITS": visits,
            "SATISFACTION": satisfaction,
            "EBOOK_DEVICE": device,
        })
    survey_df = pd.DataFrame(records, columns=SURVEY_COLUMNS_LIST)
    survey_df["MEMBERSHIP"] = survey_df["MEMBERSHIP"].astype("category")
    survey_df["AGE_BAND"] = pd.Categorical(survey_df["AGE_BAND"], categories=age_bands, ordered=True)
    missing_counts = survey_df.isna().sum()
    special_counts = {
        "MINUTES_PER_DAY": int((survey_df["MINUTES_PER_DAY"] == -1).sum()),
        "BRANCH_VISITS": int((survey_df["BRANCH_VISITS"] == 999).sum()),
    }
    survey_df.attrs["missing_counts"] = {key: int(value) for key, value in missing_counts.items() if value}
    survey_df.attrs["special_counts"] = special_counts
    return survey_df


survey_df = create_sample_reader_df()
display(survey_df.head())
print("Survey shape:", survey_df.shape)
print("Missing values:", survey_df.attrs["missing_counts"])
print("Special codes:", survey_df.attrs["special_counts"])

In [ ]:
KNOWN_CATEGORY_ORDER = {
    "AGE_BAND": ["16-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "MEMBERSHIP": ["Student", "Basic", "Plus", "Family"],
    "SATISFACTION": [1, 2, 3, 4, 5],
}
SPECIAL_CODES = {"MINUTES_PER_DAY": [-1], "BRANCH_VISITS": [999]}


def looks_numeric(series):
    """Return True when most non-empty values parse as numbers."""
    values = series.dropna()
    if values.empty:
        return False
    parsed = pd.to_numeric(values, errors="coerce")
    return parsed.notna().mean() >= 0.9


def tidy_label(value):
    """Turn a raw category value into a readable axis label."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "Missing"
    return str(value).strip().replace("_", " ").title()


def build_axis_for_column(frame, column_name, max_categories=8):
    """Describe how one survey column should appear on a parallel plot.

    Numeric columns keep their scale, with special codes and missing
    values moved just below the observed range; categorical columns are
    mapped to evenly spaced positions in a known or frequency order.
    """
    series = frame[column_name]
    special_values = SPECIAL_CODES.get(column_name, [])
    axis_info = {"column": column_name, "kind": None, "positions": None,
                 "tick_values": [], "tick_labels": [], "notes": []}
    if column_name in KNOWN_CATEGORY_ORDER or not looks_numeric(series):
        if column_name in KNOWN_CATEGORY_ORDER:
            order = list(KNOWN_CATEGORY_ORDER[column_name])
        else:
            counts = series.fillna("Missing").astype(str).value_counts()
            order = list(counts.index[:max_categories])
            if len(counts) > max_categories:
                axis_info["notes"].append(f"{len(counts) - max_categories} rare categories grouped as Other")
                order.append("Other")
        raw_values = series.astype(object).where(series.notna(), "Missing")
        if column_name not in KNOWN_CATEGORY_ORDER:
            raw_values = raw_values.astype(str).where(raw_values.astype(str).isin(order), "Other")
        if "Missing" in set(raw_values.astype(str)) and "Missing" not in order:
            order.append("Missing")
        position_map = {value: index for index, value in enumerate(order)}
        positions = raw_values.map(lambda value: position_map.get(value, position_map.get(str(value), np.nan)))
        axis_info["kind"] = "category"
        axis_info["positions"] = positions.astype(float)
        axis_info["tick_values"] = list(range(len(order)))
        axis_info["tick_labels"] = [tidy_label(value) for value in order]
        return axis_info
    numeric = pd.to_numeric(series, errors="coerce")
    special_mask = numeric.isin(special_values)
    missing_mask = numeric.isna()
    valid = numeric[~special_mask & ~missing_mask]
    if valid.empty:
        axis_info["kind"] = "empty"
        axis_info["positions"] = pd.Series(np.nan, index=series.index)
        axis_info["notes"].append("no usable values")
        return axis_info
    low, high = float(valid.min()), float(valid.max())
    span = high - low if high > low else 1.0
    gap = span * 0.12
    positions = numeric.copy().astype(float)
    tick_values = list(np.linspace(low, high, 5))
    tick_labels = [f"{value:.1f}" if span < 20 else f"{value:.0f}" for value in tick_values]
    next_slot = low - gap
    for code in special_values:
        if special_mask.any() and (numeric == code).any():
            positions[numeric == code] = next_slot
            tick_values.insert(0, next_slot)
            tick_labels.insert(0, f"code {code}")
            axis_info["notes"].append(f"{int((numeric == code).sum())} values coded {code}")
            next_slot -= gap
    if missing_mask.any():
        positions[missing_mask] = next_slot
        tick_values.insert(0, next_slot)
        tick_labels.insert(0, "Missing")
        axis_info["notes"].append(f"{int(missing_mask.sum())} missing values")
    axis_info["kind"] = "numeric"
    axis_info["positions"] = positions
    axis_info["tick_values"] = tick_values
    axis_info["tick_labels"] = tick_labels
    return axis_info

In [ ]:
def draw_parallel_axes(frame, columns, color_column=None, title_text="Reader survey"):
    """Draw a simple parallel-coordinates chart and return a summary."""
    axes_info = [build_axis_for_column(frame, column_name) for column_name in columns]
    normalized = []
    for axis_info in axes_info:
        positions = axis_info["positions"].astype(float)
        tick_values = axis_info["tick_values"] or [0.0]
        low, high = min(tick_values), max(tick_values)
        span = high - low if high > low else 1.0
        normalized.append((positions - low) / span)
    stacked = np.vstack([values.to_numpy() for values in normalized])
    figure, axis = plt.subplots(figsize=(11, 4))
    if color_column is not None:
        color_values = pd.to_numeric(frame[color_column], errors="coerce")
        color_low, color_high = color_values.min(), color_values.max()
        color_span = color_high - color_low if color_high > color_low else 1.0
        colors = plt.cm.viridis(((color_values - color_low) / color_span).fillna(0).to_numpy())
    else:
        colors = ["steelblue"] * stacked.shape[1]
    drawn_lines = 0
    for column_index in range(stacked.shape[1]):
        line_values = stacked[:, column_index]
        if np.isnan(line_values).all():
            continue
        axis.plot(range(len(columns)), line_values, color=colors[column_index], alpha=0.5, linewidth=1)
        drawn_lines += 1
    for axis_index, axis_info in enumerate(axes_info):
        axis.axvline(axis_index, color="black", linewidth=0.8)
        tick_values = axis_info["tick_values"] or [0.0]
        low, high = min(tick_values), max(tick_values)
        span = high - low if high > low else 1.0
        for tick_value, tick_label in zip(tick_values, axis_info["tick_labels"]):
            axis.text(axis_index + 0.03, (tick_value - low) / span, tick_label, fontsize=6)
    axis.set_xticks(range(len(columns)))
    axis.set_xticklabels([column_name.replace("_", " ").title() for column_name in columns], rotation=20)
    axis.set_yticks([])
    axis.set_title(title_text)
    plt.tight_layout()
    notes = {axis_info["column"]: axis_info["notes"] for axis_info in axes_info if axis_info["notes"]}
    summary = {
        "axes": len(axes_info),
        "lines": drawn_lines,
        "categorical_axes": sum(axis_info["kind"] == "category" for axis_info in axes_info),
        "numeric_axes": sum(axis_info["kind"] == "numeric" for axis_info in axes_info),
        "notes": notes,
    }
    plt.close(figure)
    return summary


chart_columns = [column_name for column_name in SURVEY_COLUMNS_LIST if column_name != "READER_ID"]
chart_summary = draw_parallel_axes(survey_df, chart_columns, color_column="SATISFACTION")
print("Chart summary:", {key: value for key, value in chart_summary.items() if key != "notes"})
for column_name, column_notes in chart_summary["notes"].items():
    print(f"  {column_name}: {'; '.join(column_notes)}")

In [ ]:
reading_columns = ["AGE_BAND", "BOOKS_PER_MONTH", "MINUTES_PER_DAY", "SATISFACTION"]
focus_summary = draw_parallel_axes(survey_df, reading_columns, color_column="BOOKS_PER_MONTH", title_text="Reading intensity")
print("Focus chart:", focus_summary["axes"], "axes,", focus_summary["lines"], "readers")
valid_minutes = survey_df.loc[survey_df["MINUTES_PER_DAY"] >= 0, "MINUTES_PER_DAY"]
print(f"Median minutes per day (codes excluded): {valid_minutes.median():.1f}")

In [ ]:
def add_survey_column(frame, column_name, values, position=None):
    """
    Add a new survey column without changing the original frame.

    Parameters
    ----------
    frame : pandas DataFrame
        Survey table.
    column_name : str
        Name of the new column.
    values : list, numpy array, pandas Series or scalar
        One value per reader; a scalar is repeated for every reader.
    position : int or None
        Where to insert the column; None appends it at the end.
    """
    updated = frame.copy()
    if np.isscalar(values):
        values = [values] * len(updated)
    values = list(values)
    if len(values) != len(updated):
        raise ValueError(
            f"Expected {len(updated)} values for '{column_name}', "
            f"received {len(values)}."
        )
    if position is None or position >= len(updated.columns):
        updated[column_name] = values
    else:
        updated.insert(position, column_name, values)
    return updated

In [ ]:
extra_rng = np.random.default_rng(77)
podcast_hours = extra_rng.choice([np.nan, 0.0, 0.5, 1.0, 2.0, 3.5], size=len(survey_df), p=[0.08, 0.3, 0.2, 0.2, 0.15, 0.07])
survey_with_podcasts_df = add_survey_column(survey_df, "PODCAST_HOURS", podcast_hours, position=None)
display(survey_with_podcasts_df.head())
print("Columns now:", len(survey_with_podcasts_df.columns))

In [ ]:
extended_summary = draw_parallel_axes(survey_with_podcasts_df, chart_columns + ["PODCAST_HOURS"], color_column="SATISFACTION", title_text="With podcast hours")
print("Extended chart axes:", extended_summary["axes"])

In [ ]:
survey_with_region_df = add_survey_column(survey_with_podcasts_df, "REGION", "North", position=1)
print("Region column position:", list(survey_with_region_df.columns).index("REGION"))